In [ ]:
!pip install opencv-python-headless
!pip install scikit-learn
!pip install efficientnet

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt
import glob

from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score,roc_auc_score, f1_score,confusion_matrix, ConfusionMatrixDisplay

from keras.applications import VGG16, ResNet50
from keras.applications.efficientnet import EfficientNetB0
from keras.applications.vgg16 import preprocess_input as preprocess_vgg
from keras.applications.resnet import preprocess_input as preprocess_resnet
from keras.applications.efficientnet import preprocess_input as preprocess_eff
from keras.models import Model

import tensorflow as tf
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_eff

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# paths
faceforensis_real_path = "/kaggle/input/faceforensics-face/REAL"
faceforensis_fake_deepfake = "/kaggle/input/faceforensics-face/Deepfake"
faceforensis_fake_deepfakedetection = "/kaggle/input/faceforensics-face/DeepfakeDetection"
faceforensis_fake_face2face = "/kaggle/input/faceforensics-face/Face2Face"
faceforensis_fake_faceshifter = "/kaggle/input/faceforensics-face/FaceShifter"
faceforensis_fake_faceswap = "/kaggle/input/faceforensics-face/FaceSwap"
faceforensis_fake_neuraltextures = "/kaggle/input/faceforensics-face/NeuralTextures"

# models
vgg_model = VGG16(weights='imagenet', include_top=False, pooling='avg')
resnet_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
eff_model = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')

2025-05-01 12:43:21.252637: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
def plot_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["REAL", "FAKE"])
    plt.figure(figsize=(5, 5))
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.show()

In [5]:
def extract_features_from_image(image_path): 
    # Verifica se o arquivo é uma imagem válida
    if not os.path.isfile(image_path):
        print(f"[ERRO] O arquivo {image_path} não é válido ou não é uma imagem.")
        return None

    # Carrega a imagem
    img = cv2.imread(image_path)
    if img is None:
        print(f"[ERRO] Não foi possível carregar a imagem {image_path}.")
        return None

    # Redimensiona a imagem
    img_resized = cv2.resize(img, (224, 224))
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

    # Processamento para VGG, ResNet e EfficientNet
    img_batch_vgg = preprocess_vgg(np.expand_dims(img_rgb.astype('float32'), axis=0))
    img_batch_resnet = preprocess_resnet(np.expand_dims(img_rgb.astype('float32'), axis=0))
    img_batch_eff = preprocess_eff(np.expand_dims(img_rgb.astype('float32'), axis=0))

    vgg_feat = vgg_model.predict(img_batch_vgg, verbose=0).flatten()
    res_feat = resnet_model.predict(img_batch_resnet, verbose=0).flatten()
    eff_feat = eff_model.predict(img_batch_eff, verbose=0).flatten()

    combined = np.concatenate([vgg_feat, res_feat, eff_feat])

    return combined  # Retorne a combinação das features

def generate_model(real_path, fake_path):
    X_train, y_train = [], []
    X_test, y_test = [], []

    for label, path in [("REAL", real_path), ("FAKE", fake_path)]:
        image_files = os.listdir(path)
        test_files = image_files[:50]
        train_files = image_files[50:]

        for image_name in tqdm(test_files, desc=f"{label} (TESTE)"):
            image_path = os.path.join(path, image_name)
            feat = extract_features_from_image(image_path)
            if feat is not None:
                X_test.append(feat)
                y_test.append(0 if label == "REAL" else 1)
            else:
                print(f"[INFO] Não foram extraídas características de {image_path}")

        for image_name in tqdm(train_files, desc=f"{label} (TREINO)"):
            image_path = os.path.join(path, image_name)
            feat = extract_features_from_image(image_path)
            if feat is not None:
                X_train.append(feat)
                y_train.append(0 if label == "REAL" else 1)
            else:
                print(f"[INFO] Não foram extraídas características de {image_path}")

    if len(X_train) == 0 or len(X_test) == 0:
        print("Erro: Não foram extraídas características suficientes para o treino ou teste.")
        return None, None

    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)

    # PCA + SVM
    pca = PCA(n_components=10)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    svm = SVC(kernel='linear')
    svm.fit(X_train_pca, y_train)

    y_pred = svm.predict(X_test_pca)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1-score (macro):", f1_score(y_test, y_pred, average='macro'))

    if hasattr(svm, "decision_function"):
        y_scores = svm.decision_function(X_test_pca)
        auc = roc_auc_score(y_test, y_scores)
        print("AUC:", auc)
    else:
        print("O modelo SVM não possui decision_function para calcular a AUC.")

    print("Classification Report:\n", classification_report(y_test, y_pred))

    return svm, pca

# Chamada da função
svm, pca = generate_model(faceforensis_real_path, faceforensis_fake_deepfake)


FAKE (TREINO): 100%|██████████| 200/200 [02:16<00:00,  1.47it/s]


Accuracy: 0.71
F1-score (macro): 0.7092731829573935
AUC: 0.7292
Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.76      0.72        50
           1       0.73      0.66      0.69        50

    accuracy                           0.71       100
   macro avg       0.71      0.71      0.71       100
weighted avg       0.71      0.71      0.71       100

